In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

print("KRYPT Model Improvement Environment Ready!")

KRYPT Model Improvement Environment Ready!


In [2]:
import json

with open("../data/bitcoin_funding.json", "r") as file:
    raw_data = json.load(file)

df = pd.DataFrame(raw_data)

print("Dataset loaded!")
print("Shape:", df.shape)
print(df.head())

Dataset loaded!
Shape: (500, 5)
    symbol    fundingTime fundingRate       markPrice rateType
0  BTCUSDT  1780156800000  0.00006214  73892.79565217  Regular
1  BTCUSDT  1780185600000  0.00004432  73856.90000000  Regular
2  BTCUSDT  1780214400000  0.00005663  73833.30000000  Regular
3  BTCUSDT  1780243200004  0.00005988  73613.40000000  Regular
4  BTCUSDT  1780272000001  0.00005703  73653.56663043  Regular


In [3]:
df["fundingRate"] = pd.to_numeric(df["fundingRate"])
df["markPrice"] = pd.to_numeric(df["markPrice"])

df["fundingTime"] = pd.to_datetime(
    df["fundingTime"], unit="ms"
)

# Create model features
df["funding_rate_change"] = df["fundingRate"].diff()
df["mark_price_return"] = df["markPrice"].pct_change()

# Create next-period prediction target
df["next_funding_rate"] = df["fundingRate"].shift(-1)

# Same provisional spike threshold as Random Forest
df["target_spike"] = (
    df["next_funding_rate"] >= 0.0001
).astype(int)

# Remove incomplete rows
df = df.dropna().reset_index(drop=True)

FEATURES = [
    "fundingRate",
    "markPrice",
    "funding_rate_change",
    "mark_price_return"
]

X = df[FEATURES]
y = df["target_spike"]

print("Features and target prepared!")
print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features and target prepared!
Dataset shape: (498, 9)

Target distribution:
target_spike
0    420
1     78
Name: count, dtype: int64


In [4]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training spike cases:", y_train.sum())
print("Testing spike cases:", y_test.sum())

Training records: 398
Testing records: 100
Training spike cases: 48
Testing spike cases: 30


In [5]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")

XGBoost training completed!


In [6]:
y_pred = xgb_model.predict(X_test)

print("XGBoost Model Performance")
print("-------------------------")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

XGBoost Model Performance
-------------------------
Accuracy: 0.79
Precision: 0.7647058823529411
Recall: 0.43333333333333335
F1 Score: 0.5531914893617021

Confusion Matrix:
[[66  4]
 [17 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.94      0.86        70
           1       0.76      0.43      0.55        30

    accuracy                           0.79       100
   macro avg       0.78      0.69      0.71       100
weighted avg       0.79      0.79      0.77       100



In [7]:
joblib.dump(
    xgb_model,
    "../models/xgboost_model.pkl"
)

print("XGBoost model saved successfully!")

XGBoost model saved successfully!


In [8]:
scale = (y_train == 0).sum() / (y_train == 1).sum()

xgb_weighted = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    eval_metric="logloss",
    random_state=42
)

xgb_weighted.fit(X_train, y_train)

print("Class-weighted XGBoost training completed!")
print("Class weight:", scale)

Class-weighted XGBoost training completed!
Class weight: 7.291666666666667


In [9]:
y_pred_weighted = xgb_weighted.predict(X_test)

print("Weighted XGBoost Performance")
print("----------------------------")

print("Accuracy:", accuracy_score(y_test, y_pred_weighted))
print("Precision:", precision_score(y_test, y_pred_weighted, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_weighted, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred_weighted, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_weighted))

Weighted XGBoost Performance
----------------------------
Accuracy: 0.74
Precision: 0.5588235294117647
Recall: 0.6333333333333333
F1 Score: 0.59375

Confusion Matrix:
[[55 15]
 [11 19]]


In [10]:
joblib.dump(
    xgb_weighted,
    "../models/xgboost_weighted_experiment.pkl"
)

print("Weighted XGBoost experiment saved!")

Weighted XGBoost experiment saved!


In [11]:
validation_index = int(len(X_train) * 0.8)

X_subtrain = X_train.iloc[:validation_index]
X_val = X_train.iloc[validation_index:]

y_subtrain = y_train.iloc[:validation_index]
y_val = y_train.iloc[validation_index:]

print("Sub-training records:", len(X_subtrain))
print("Validation records:", len(X_val))
print("Validation spike cases:", y_val.sum())

Sub-training records: 318
Validation records: 80
Validation spike cases: 7


In [12]:
val_weight = (y_subtrain == 0).sum() / (y_subtrain == 1).sum()

xgb_tuning = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=val_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_tuning.fit(X_subtrain, y_subtrain)

val_probabilities = xgb_tuning.predict_proba(X_val)[:, 1]

print("Validation model trained!")
print("Validation probabilities generated!")

Validation model trained!
Validation probabilities generated!


In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

for threshold in thresholds:
    predictions = (val_probabilities >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", round(
        precision_score(y_val, predictions, zero_division=0), 3
    ))
    print("Recall:", round(
        recall_score(y_val, predictions, zero_division=0), 3
    ))
    print("F1:", round(
        f1_score(y_val, predictions, zero_division=0), 3
    ))


Threshold: 0.3
Precision: 0.16
Recall: 0.571
F1: 0.25

Threshold: 0.35
Precision: 0.167
Recall: 0.571
F1: 0.258

Threshold: 0.4
Precision: 0.15
Recall: 0.429
F1: 0.222

Threshold: 0.45
Precision: 0.167
Recall: 0.429
F1: 0.24

Threshold: 0.5
Precision: 0.143
Recall: 0.286
F1: 0.19

Threshold: 0.55
Precision: 0.0
Recall: 0.0
F1: 0.0

Threshold: 0.6
Precision: 0.0
Recall: 0.0
F1: 0.0


In [14]:
import json

with open("../data/xrp.json", "r") as file:
    xrp_data = json.load(file)

print("XRP records:", len(xrp_data))
print("First record:")
print(xrp_data[0])

JSONDecodeError: Expecting ',' delimiter: line 7002 column 3 (char 148387)

In [15]:
import json

with open("../data/xrp.json", "r") as file:
    xrp_data = json.load(file)

print("XRP records:", len(xrp_data))
print("First record:", xrp_data[0])

XRP records: 1726
First record: {'symbol': 'XRPUSDT', 'fundingTime': 1640995200006, 'fundingRate': '0.00010000', 'markPrice': '', 'rateType': 'Regular'}


In [16]:
xrp_df = pd.DataFrame(xrp_data)

print("Total records:", len(xrp_df))

print("\nMissing values:")
print(xrp_df.isin(["", None]).sum())

print("\nDuplicate timestamps:", 
      xrp_df["fundingTime"].duplicated().sum())

print("\nData types:")
print(xrp_df.dtypes)

Total records: 1726

Missing values:
symbol            0
fundingTime       0
fundingRate       0
markPrice      1726
rateType          0
dtype: int64

Duplicate timestamps: 88

Data types:
symbol           str
fundingTime    int64
fundingRate      str
markPrice        str
rateType         str
dtype: object


In [17]:
duplicates = xrp_df[
    xrp_df.duplicated(subset=["fundingTime"], keep=False)
]

print("Duplicate records:", len(duplicates))

print(duplicates.head(10).to_string(index=False))

Duplicate records: 176
 symbol   fundingTime fundingRate markPrice rateType
XRPUSDT 1667260800000 -0.00014557            Regular
XRPUSDT 1667289600000  0.00010000            Regular
XRPUSDT 1667318400000  0.00010000            Regular
XRPUSDT 1667347200000 -0.00003503            Regular
XRPUSDT 1667376000000  0.00002174            Regular
XRPUSDT 1667404800019 -0.00003512            Regular
XRPUSDT 1667433600011 -0.00016340            Regular
XRPUSDT 1667462400016  0.00000971            Regular
XRPUSDT 1667491200005  0.00010000            Regular
XRPUSDT 1667520000000 -0.00006566            Regular


In [18]:
duplicate_check = (
    xrp_df[xrp_df.duplicated("fundingTime", keep=False)]
    .groupby("fundingTime")
    .agg(
        record_count=("fundingRate", "size"),
        unique_rates=("fundingRate", "nunique"),
        rates=("fundingRate", lambda x: list(x))
    )
)

print(duplicate_check.head(15).to_string())

               record_count  unique_rates                       rates
fundingTime                                                          
1667260800000             2             1  [-0.00014557, -0.00014557]
1667289600000             2             1    [0.00010000, 0.00010000]
1667318400000             2             1    [0.00010000, 0.00010000]
1667347200000             2             1  [-0.00003503, -0.00003503]
1667376000000             2             1    [0.00002174, 0.00002174]
1667404800019             2             1  [-0.00003512, -0.00003512]
1667433600011             2             1  [-0.00016340, -0.00016340]
1667462400016             2             1    [0.00000971, 0.00000971]
1667491200005             2             1    [0.00010000, 0.00010000]
1667520000000             2             1  [-0.00006566, -0.00006566]
1667548800004             2             1    [0.00009885, 0.00009885]
1667577600020             2             1    [0.00010000, 0.00010000]
1667606400009       

In [19]:
# Remove exact duplicate records
xrp_clean = xrp_df.drop_duplicates().copy()

print("Original records:", len(xrp_df))
print("Records after cleaning:", len(xrp_clean))
print("Removed:", len(xrp_df) - len(xrp_clean))

print("\nRemaining duplicate timestamps:",
      xrp_clean["fundingTime"].duplicated().sum())

Original records: 1726
Records after cleaning: 1638
Removed: 88

Remaining duplicate timestamps: 0


In [20]:
# Convert funding rate to numeric
xrp_clean["fundingRate"] = pd.to_numeric(
    xrp_clean["fundingRate"], errors="coerce"
)

# Convert timestamps to datetime
xrp_clean["fundingTime"] = pd.to_datetime(
    xrp_clean["fundingTime"], unit="ms"
)

# Sort chronologically
xrp_clean = xrp_clean.sort_values("fundingTime").reset_index(drop=True)

print(xrp_clean.dtypes)
print("\nDate range:")
print("Start:", xrp_clean["fundingTime"].min())
print("End:", xrp_clean["fundingTime"].max())

print("\nFunding rate summary:")
print(xrp_clean["fundingRate"].describe())


symbol                    str
fundingTime    datetime64[us]
fundingRate           float64
markPrice                 str
rateType                  str
dtype: object

Date range:
Start: 2022-01-01 00:00:00.006000
End: 2023-06-30 16:00:00.001000

Funding rate summary:
count    1638.000000
mean        0.000015
std         0.000121
min        -0.001394
25%        -0.000037
50%         0.000060
75%         0.000100
max         0.000100
Name: fundingRate, dtype: float64
